In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame()

In [ ]:
import unicodedata
import re
from unidecode import unidecode

def normalize_name(name):

    name = name.lower()

    name = unidecode(name)

    name = re.sub(r'[^\w\s]', '', name) 
    name = re.sub(r'\s+', ' ', name).strip()  
    

    replacements = {
        'jr': '', 'sr': '', 'dr': '', 'lic': '', 'ing': '', 
        'mtro': '', 'phd': '', 'c': '', 's a': '', 's.a.': ''
    }
    
    for k, v in replacements.items():
        name = name.replace(k, v)
    
    return name


normalize_name("Dr. Juan Pérez-García, Jr.")  # Resultado: "juan perez garcia"







In [ ]:
from jellyfish import jaro_winkler_similarity, levenshtein_distance, soundex, metaphone
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def advanced_features(name1, name2):
    n1 = normalize_name(name1)
    n2 = normalize_name(name2)
    
    # Dividir en tokens
    tokens1 = n1.split()
    tokens2 = n2.split()
    
    # Características básicas
    features = {
        'jaro_winkler': jaro_winkler_similarity(n1, n2),
        'levenshtein': levenshtein_distance(n1, n2),
        'length_diff': abs(len(n1) - len(n2)),
        'length_ratio': min(len(n1), len(n2)) / max(len(n1), len(n2), 1),
        'same_soundex': int(soundex(n1) == soundex(n2)),
        'same_metaphone': int(metaphone(n1) == metaphone(n2)),
    }
    
    # Características de tokens
    common_tokens = set(tokens1) & set(tokens2)
    features.update({
        'common_token_count': len(common_tokens),
        'token_ratio': len(common_tokens) / max(len(set(tokens1 + tokens2)), 1),
        'token_jaccard': len(common_tokens) / len(set(tokens1 + tokens2)) if tokens1 or tokens2 else 0,
        'first_token_match': int(tokens1[0] == tokens2[0]) if tokens1 and tokens2 else 0,
        'last_token_match': int(tokens1[-1] == tokens2[-1]) if tokens1 and tokens2 else 0,
    })
    
    # Características de iniciales
    initials1 = ''.join([t[0] for t in tokens1 if t])
    initials2 = ''.join([t[0] for t in tokens2 if t])
    features.update({
        'initials_match': int(initials1 == initials2),
        'initials_jaro': jaro_winkler_similarity(initials1, initials2),
    })
    
    # Características de orden de tokens (para nombres invertidos)
    features['inverted_name'] = int(
        len(tokens1) == 2 and len(tokens2) == 2 and 
        tokens1[0] == tokens2[1] and tokens1[1] == tokens2[0]
    )
    
    return features

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.metrics import classification_report, make_scorer, f1_score


def prepare_data(df):
    X = []
    for _, row in df.iterrows():
        features = advanced_features(row['nombre1'], row['nombre2'])
        X.append(features)
    
    feature_df = pd.DataFrame(X)
    return feature_df, df['etiqueta']


X, y = prepare_data(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), ['levenshtein', 'length_diff']),
    ('minmax', MinMaxScaler(), ['jaro_winkler', 'length_ratio', 'token_ratio', 'token_jaccard'])
], remainder='passthrough')

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])


param_grid = {
    'classifier': [RandomForestClassifier(), GradientBoostingClassifier()],
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 5, 10],
    'classifier__min_samples_split': [2, 5],
}


grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring=make_scorer(f1_score),
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns


y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precisión-Recall')
plt.show()

In [ ]:
import joblib


joblib.dump(best_model, 'model.pkl')


loaded_model = joblib.load('model.pkl')


def predict_if_same_name(name1, name2, model=loaded_model, threshold=0.5):
    features = advanced_features(name1, name2)
    features_df = pd.DataFrame([features])
    
    features_df = features_df[X_train.columns]
    
    proba = model.predict_proba(features_df)[0][1]
    prediction = int(proba >= threshold)
    
    return {
        'prediction': prediction,
        'probability': proba,
        'features': features
    }


predict_if_same_name("Carlos Ruiz", "C. Ruiz")